# Django Management Commands & Logging

## What are Management Commands?

Management commands are Python scripts that run via `python manage.py <command>`. Django ships with many built-in commands (`migrate`, `createsuperuser`, `shell`, etc.). You can write custom commands to automate tasks such as data imports, scheduled jobs, or maintenance scripts.


## Creating a Custom Management Command

### Directory structure

```
myapp/
    management/
        __init__.py
        commands/
            __init__.py
            send_reminders.py
```

Both `management/` and `management/commands/` must contain `__init__.py`.

### Minimal command

```python
# myapp/management/commands/send_reminders.py
from django.core.management.base import BaseCommand

class Command(BaseCommand):
    help = 'Send reminder emails to users with pending orders'

    def handle(self, *args, **options):
        self.stdout.write('Sending reminders...')
        # ... business logic here ...
        self.stdout.write(self.style.SUCCESS('Done.'))
```

Run it:
```bash
python manage.py send_reminders
```


## Adding Arguments

```python
class Command(BaseCommand):
    help = 'Send reminders for a specific date range'

    def add_arguments(self, parser):
        # positional argument
        parser.add_argument('days', type=int, help='Number of past days to check')
        # optional flag
        parser.add_argument('--dry-run', action='store_true',
                            help='Print what would be done without sending emails')

    def handle(self, *args, **options):
        days    = options['days']
        dry_run = options['dry_run']
        self.stdout.write(f'Checking last {days} days. Dry run: {dry_run}')
```

```bash
python manage.py send_reminders 7
python manage.py send_reminders 7 --dry-run
```


## stdout, stderr, and style

| Method | Output |
|--------|--------|
| `self.stdout.write(msg)` | Normal output |
| `self.stderr.write(msg)` | Error output |
| `self.style.SUCCESS(msg)` | Green text |
| `self.style.WARNING(msg)` | Yellow text |
| `self.style.ERROR(msg)` | Red text |
| `self.style.NOTICE(msg)` | Cyan text |

```python
def handle(self, *args, **options):
    try:
        result = process_orders()
        self.stdout.write(self.style.SUCCESS(f'Processed {result} orders.'))
    except Exception as e:
        self.stderr.write(self.style.ERROR(f'Failed: {e}'))
        raise SystemExit(1)
```

Exit code `1` indicates failure to shell scripts and CI pipelines.


## Calling One Command from Another

```python
from django.core.management import call_command

class Command(BaseCommand):
    help = 'Full nightly refresh'

    def handle(self, *args, **options):
        call_command('import_products')
        call_command('send_reminders', 7, dry_run=False)
        self.stdout.write(self.style.SUCCESS('Nightly refresh complete.'))
```


## Django Logging

Django uses Python's built-in `logging` module. Configure it in `settings.py` via the `LOGGING` dict.

### Basic configuration

```python
# settings.py
LOGGING = {
    'version': 1,
    'disable_existing_loggers': False,

    'formatters': {
        'verbose': {
            'format': '{levelname} {asctime} {module} {message}',
            'style': '{',
        },
    },

    'handlers': {
        'console': {
            'class': 'logging.StreamHandler',
            'formatter': 'verbose',
        },
        'file': {
            'class': 'logging.FileHandler',
            'filename': BASE_DIR / 'logs/django.log',
            'formatter': 'verbose',
        },
    },

    'root': {
        'handlers': ['console'],
        'level': 'WARNING',
    },

    'loggers': {
        'django': {
            'handlers': ['console', 'file'],
            'level': 'INFO',
            'propagate': False,
        },
        'myapp': {
            'handlers': ['console', 'file'],
            'level': 'DEBUG',
            'propagate': False,
        },
    },
}
```


## Using a Logger in Code

```python
import logging

logger = logging.getLogger(__name__)

def process_payment(order_id, amount):
    logger.debug('Processing payment for order %s', order_id)
    try:
        result = charge_card(amount)
        logger.info('Payment successful for order %s: %s', order_id, result)
        return result
    except PaymentError as e:
        logger.error('Payment failed for order %s: %s', order_id, e, exc_info=True)
        raise
```

Log levels (lowest to highest):

| Level | When to use |
|-------|-------------|
| `DEBUG` | Detailed diagnostic info |
| `INFO` | Confirmation that things are working |
| `WARNING` | Something unexpected, but the app continues |
| `ERROR` | A serious problem occurred |
| `CRITICAL` | The application cannot continue |

Use `exc_info=True` to include the full traceback in the log record.


## Logging Inside Management Commands

Combine good logging with management commands so automated runs are auditable:

```python
import logging
from django.core.management.base import BaseCommand
from orders.models import Order

logger = logging.getLogger(__name__)

class Command(BaseCommand):
    help = 'Cancel orders unpaid for more than 24 hours'

    def handle(self, *args, **options):
        stale = Order.objects.filter(status='pending', created__lt=cutoff)
        count = stale.count()
        logger.info('Found %d stale orders to cancel', count)

        stale.update(status='cancelled')
        logger.info('Cancelled %d orders', count)
        self.stdout.write(self.style.SUCCESS(f'Cancelled {count} orders.'))
```


## Summary

- Custom management commands live in `myapp/management/commands/<name>.py` and implement a `Command` class with a `handle()` method.
- Use `add_arguments()` to accept positional arguments and optional flags.
- Write to `self.stdout` / `self.stderr` and use `self.style.*` for colored output.
- `call_command()` lets you invoke other management commands programmatically.
- Django logging is configured in `settings.py` with handlers, formatters, and loggers.
- Create per-module loggers with `logging.getLogger(__name__)` to trace where a message originates.
- Use `exc_info=True` with `logger.error()` to capture full tracebacks automatically.
